In [1]:
import pandas as pd
import os
import re

# This isn't included in the default container,
# you'll have to install it from pip
import markdown

In [77]:
ROOT_PATH = "../../.ignore/terraform/website/docs/language/functions/"
files = os.listdir(ROOT_PATH)
forward = {}
backward = {}

# for filename in files[5:6]:
for filename in files:
    with open(f"{ROOT_PATH}/{filename}", "r") as f_open:
        content = f_open.read()

        content = re.sub(r"---.*---\n\n", "", content, flags=re.DOTALL)
        title = content.split("\n")[0]
        rest = "\n".join(content.split("\n")[1:]).lstrip()
        rest = re.sub(r"->.*\n\n", "", rest)
        rest = re.sub(r"`.*` ", "", rest, count=1)
        rest = rest.capitalize()
        first_line = re.sub("\n", " ", rest.split("\n\n")[0])

        rest_html = markdown.markdown(rest, extensions=["fenced_code"])
        title_html = markdown.markdown(title, extensions=["fenced_code"])

        key = re.sub(r"\..*$", "", filename)

        forward[key] = rest_html
        backward[first_line] = f"{title_html}\n\n{rest_html}"

In [78]:
forward_df = pd.DataFrame(data=forward.values(), index=forward.keys())
backward_df = pd.DataFrame(data=backward.values(), index=backward.keys())
merged = pd.concat([forward_df, backward_df])
merged

,0
join,<p>Produces a string by concatenating all of t...
lookup,<p>Retrieves the value of a single element fro...
urlencode,<p>Applies url encoding to a given string.</p>...
fileexists,<p>Determines whether a file exists at a given...
yamlencode,<p>Encodes a given value to a string using\n<a...
...,...
"The function is no longer available. prior to terraform v0.12 it was the only available syntax for writing a literal list inside an expression, but terraform v0.12 introduced a new first-class syntax.",<h1><code>list</code> Function</h1>\n\n<p>The ...
Is a variant of [`sha512`](/terraform/language/functions/sha512) that hashes the contents of a given file rather than a literal string.,<h1><code>filesha512</code> Function</h1>\n\n<...
Produces a list by dividing a given string at all occurrences of a given separator.,<h1><code>split</code> Function</h1>\n\n<p>Pro...
Is a rarely-needed function which takes a string containing the content of a [`.tfvars` file](/terraform/language/values/variables#variable-definitions-tfvars-files) and returns an object describing the raw variable values it defines.,<h1><code>provider::terraform::decode_tfvars</...


In [79]:
merged.to_csv(
    "../../.ignore/terraform-functions.csv",
    encoding="utf-8",
    sep=";",
    header=False,
)